# 3. 비만 위험도 예측 모델 (Prediction)

**목적:** 행동·생활습관 피처로 비만 여부(is_obese)를 이진 분류합니다.

**중요:** `Height`, `Weight`, `BMI`를 피처에서 완전히 제외합니다.
실제 의료 현장에서 '습관 정보만으로 비만 위험을 얼마나 예측할 수 있는가'를 평가합니다.
이로 인해 정확도는 99%가 아닌 **현실적인 수치**가 나옵니다.

**입력:** `artifacts/processed_data.csv`, `artifacts/encoders.pkl`, `artifacts/scaler.pkl`
**출력:** `artifacts/prediction_model.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc
)

ARTIFACTS = '../artifacts'

In [ ]:
df = pd.read_csv(f'{ARTIFACTS}/processed_data.csv')

CATEGORICAL_FEATURES = [
    'Gender', 'CALC', 'FAVC', 'SCC', 'SMOKE',
    'family_history_with_overweight', 'CAEC', 'MTRANS'
]
CONTINUOUS_FEATURES = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
BEHAVIORAL_FEATURES = CATEGORICAL_FEATURES + CONTINUOUS_FEATURES

# 이진 타겟: 비만 3유형 → 1, 나머지 → 0
obese_labels = ['Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III']
df['is_obese'] = df['NObeyesdad'].isin(obese_labels).astype(int)

print('타겟 분포:')
print(df['is_obese'].value_counts())
print(f'비만 비율: {df["is_obese"].mean():.2%}')

## 학습/테스트 분할

In [ ]:
X = df[BEHAVIORAL_FEATURES]
y = df['is_obese']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'학습: {len(X_train)} / 테스트: {len(X_test)}')

## 모델 학습 (Logistic Regression)

In [ ]:
clf = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',  # 클래스 불균형 보정
    random_state=42
)
clf.fit(X_train, y_train)

# 5-fold 교차 검증
cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='roc_auc')
print(f'5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 성능 평가

In [ ]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f'정확도 (Accuracy):  {acc:.4f}')
print(f'ROC AUC:           {roc_auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not Obese', 'Obese']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Not Obese', 'Obese']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'navy',linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## 피처 중요도 (회귀 계수)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': BEHAVIORAL_FEATURES,
    'Coefficient': clf.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['Coefficient']]
sns.barplot(x='Coefficient', y='Feature', data=coef_df, palette=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Feature Importance\n(양수=비만 위험 증가, 음수=감소)')
plt.tight_layout()
plt.show()

display(coef_df)

## 모델 저장

In [ ]:
with open(f'{ARTIFACTS}/prediction_model.pkl', 'wb') as f:
    pickle.dump({
        'model': clf,
        'features': BEHAVIORAL_FEATURES,
        'obese_labels': obese_labels
    }, f)
print('prediction_model.pkl 저장 완료')

## 비만 위험도 예측 함수

저장된 인코더, 스케일러를 재사용하여 새로운 입력값 예측

In [ ]:
def predict_obesity_risk(user_input: dict) -> dict:
    """
    Parameters
    ----------
    user_input : dict
        행동·생활습관 정보 (Height, Weight 불필요)
        필수 키: Age, Gender, CALC, FAVC, FCVC, NCP, SCC,
                 SMOKE, CH2O, family_history_with_overweight,
                 FAF, TUE, CAEC, MTRANS

    Returns
    -------
    dict with risk_score (0~1) and label
    """
    with open(f'{ARTIFACTS}/encoders.pkl', 'rb') as f:
        encoders = pickle.load(f)
    with open(f'{ARTIFACTS}/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    with open(f'{ARTIFACTS}/prediction_model.pkl', 'rb') as f:
        saved = pickle.load(f)
    model = saved['model']
    features = saved['features']

    row = dict(user_input)
    for col, mapping in encoders['binary'].items():
        if col in row:
            row[col] = mapping[row[col]]
    for col, order in encoders['ordinal'].items():
        if col in row:
            row[col] = {v: i for i, v in enumerate(order)}[row[col]]

    cat_feats = ['Gender','CALC','FAVC','SCC','SMOKE',
                 'family_history_with_overweight','CAEC','MTRANS']
    cont_feats = ['Age','FCVC','NCP','CH2O','FAF','TUE']
    cont_values = np.array([[row[c] for c in cont_feats]])
    scaled_cont = scaler.transform(cont_values)[0]
    scaled_row = {c: scaled_cont[i] for i, c in enumerate(cont_feats)}
    scaled_row.update({c: row[c] for c in cat_feats})

    X_new = pd.DataFrame([scaled_row])[features]
    prob = model.predict_proba(X_new)[0][1]

    if prob < 0.3:
        label, icon = '저위험', '🟢'
    elif prob < 0.6:
        label, icon = '중위험', '🟠'
    else:
        label, icon = '고위험', '🔴'

    return {'risk_score': round(float(prob), 4), 'label': label, 'icon': icon}


# 예시 입력
sample = {
    'Age': 28, 'Gender': 'Male',
    'CALC': 'Sometimes', 'FAVC': 'yes',
    'FCVC': 1, 'NCP': 3, 'SCC': 'yes', 'SMOKE': 'no',
    'CH2O': 1, 'family_history_with_overweight': 'yes',
    'FAF': 0, 'TUE': 2, 'CAEC': 'Frequently', 'MTRANS': 'Automobile'
}

result = predict_obesity_risk(sample)
print(f'{result["icon"]} 비만 위험도: {result["label"]} (score: {result["risk_score"]})')